In [19]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [20]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [27]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
# llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0)

agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

In [30]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [31]:
ask("what is the price of wireless headphones.")

[{'type': 'text', 'text': 'The price of the wireless headphones is $79.99.', 'extras': {'signature': 'EmgKZgERTTIPZ3ghEL0KTYOWFXfI2BVJqnD6KVpOrYYPpQGk/8FVzwRMwlvPXMInfpgVczRMvZz7PaOdu4Xc95kYRp7E5Zqzz5g2oD2JLhwOJP+5i5XV4U+M2G/QqV8XWm5by5it2uCo/Q=='}}]


In [32]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: str) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [33]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
# llm = ChatGroq(model="qwen/qwen3.6-27b", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
)

def ask2(question: str):
    result = agent2.invoke({
        "messages": [{"role": "user", "content": question}]
    })

    content = result["messages"][-1].content

    if isinstance(content, str):
        print(content)
    else:
        for block in content:
            if block.get("type") == "text":
                print(block["text"])

In [34]:
ask2("how do people like smart watch")

Based on 340 reviews, the smart watch has an average rating of 3.9 out of 5. This suggests that most people are generally satisfied with it, though there may be some mixed feedback.


In [35]:
ask2("what is the price and reviews of smart watch")

The smart watch is priced at **$199.99**. 

It has **340 reviews** with an average rating of **3.9 out of 5 stars** (the product details list a rating of 4.3). It features heart rate and sleep tracking, a 5-day battery life, and is water-resistant.


In [36]:
ask2("what are the reviews on this product?")

Which product are you referring to? Please provide the name of the product you'd like to see reviews for.


In [65]:
from langgraph.checkpoint.memory import InMemorySaver

# llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt="You are a helpful product assistant for an online tech store.",
    checkpointer=InMemorySaver()
)

def ask2(question: str):
    config = {
        "configurable": {
            "thread_id": "user-alice-session-1"
        }
    }

    result = agent2.invoke(
        {
            "messages": [
                {"role": "user", "content": question}
            ]
        },
        config=config
    )

    print(result["messages"][-1].text)


In [66]:
ask2("what is the price of wireless headphones.")

The wireless headphones are priced at $79.99. They feature over-ear Bluetooth connectivity, a 30-hour battery life, and active noise cancellation, with a customer rating of 4.6 stars.


In [67]:
ask2("what are the reviews on this product?")

The wireless headphones have a 4.6-star rating based on 1,262 reviews.
